In [11]:
import pandas as pd

## Overview
- Apply text classification on Vietnamese Curated Dataset, with few sample files to save resource

## EDA
- Because of subsample, make sure they have varied labels and distribution are fine 

In [12]:
input_df_1=pd.read_parquet('data/train-00000-of-00132.parquet')

In [18]:
input_df_1.shape

(92191, 3)

In [23]:
input_df_1['domain'].value_counts(dropna=False)

domain
Arts_and_Entertainment       7261
Sensitive_Subjects           6612
Health                       6546
People_and_Society           6323
Sports                       6116
News                         5970
Food_and_Drink               5125
Business_and_Industrial      4747
Travel_and_Transportation    4407
Jobs_and_Education           4256
Beauty_and_Fitness           4000
Home_and_Garden              3719
Computers_and_Electronics    3589
Law_and_Government           2897
Internet_and_Telecom         2804
Books_and_Literature         2678
Games                        2548
Autos_and_Vehicles           2508
Finance                      2446
Real_Estate                  2263
Pets_and_Animals             1532
Science                      1436
Shopping                     1310
Hobbies_and_Leisure           732
Online_Communities            366
Name: count, dtype: int64

In [14]:
input_df_2=pd.read_parquet('data/train-00001-of-00132.parquet')

In [19]:
input_df_2.shape

(92191, 3)

In [22]:
input_df_2['domain'].value_counts(dropna=False)

domain
Arts_and_Entertainment       7065
Health                       6729
People_and_Society           6354
News                         5794
Sports                       5616
Sensitive_Subjects           5323
Food_and_Drink               5179
Business_and_Industrial      4727
Travel_and_Transportation    4522
Beauty_and_Fitness           4388
Jobs_and_Education           4315
Home_and_Garden              4118
Computers_and_Electronics    3916
Books_and_Literature         3259
Internet_and_Telecom         3056
Law_and_Government           2692
Real_Estate                  2635
Autos_and_Vehicles           2492
Games                        2437
Finance                      2374
Shopping                     1475
Pets_and_Animals             1292
Science                      1276
Hobbies_and_Leisure           754
Online_Communities            403
Name: count, dtype: int64

In [20]:
[i for i in input_df_2['domain'].unique() if i not in input_df_1['domain'].unique()]

[]

In [21]:
[i for i in input_df_1['domain'].unique() if i not in input_df_2['domain'].unique()]

[]

In [15]:
len(input_df_2['domain'].unique())

25

In [40]:
print(input_df_1['text'][0])

Internet Society hay ISOC là một tổ chức quốc tế hoạt động phi lợi nhuận, phi chính phủ và bao gồm các thành viên có trình độ chuyên ngành. Tổ chức này chú trọng đến: tiêu chuẩn, giáo dục và các vấn đề về chính sách. Với trên 145 tổ chức thành viên và 65.000 thành viên cá nhân, ISOC bao gồm những con người cụ thể trong cộng đồng Internet. Mọi chi tiết có thể tìm thấy tại website của ISOC.

Internet Society nằm ở gần thủ đô Washington, DC, Hoa Kỳ và Geneva, Thụy Sĩ. Số hội viên của nó bao gồm hơn 145 tổ chức thành viên và hơn 65.000 cá nhân. Thành viên còn có thể tự lập một chi nhánh của tổ chức tùy theo vị trí hoặc sở thích. Hiện nay tổ chức có tới 90 chi nhánh trên toàn thế giới.

Nhiệm vụ và mục đích hoạt động
Bảo đảm, cổ vũ cho sự phát triển, mở rộng và sử dụng Internet được thuận lợi nhất cho mọi người trên toàn thế giới.

Xem thêm
Lịch sử Internet

Tham khảo

Liên kết ngoài
 
 
 ISOC Việt Nam
 IETF and the Internet Society - Về Internet Engineering Task Force và ISOC, bài của Vint

In [44]:
print(input_df_2['text'][1000])

Bón NPK-S Lâm Thao cho cây cà rốt?
Chủ nhật, 18/03/2018 05:04 GMT+7
10/12/2013, 11:40 (GMT+7)
Để tạo năng suất như trong SX, cây cà rốt lấy đi từ đất lượng đạm thuộc loại nhiều (chia làm 5 nhóm rau, cây cà rốt ở nhóm đầu), kali và lân thuộc loại trung bình thấp (nhóm thứ 3 của 4 nhóm).
Cây cà rốt có tên khoa học là Daucus carota var. sativa thuộc họ Hoa tán Umbelliferae.
Cây cà rốt được trồng đầu tiên ở Afghanistan và sau đó được trồng ở 10 - 12 nước khu vực phía Đông vùng Địa Trung Hải, rồi chuyển ra phía Tây khoảng 14 - 15 nước Châu Âu. Cây cà rốt được sử dụng từ lâu, đầu tiên được coi là cây thuốc, mãi cho đến những năm đầu thế kỷ XX mới được sử dụng làm cây thực phẩm. Cà rốt nổi tiếng vì là nguồn dồi dào vitaminA, hàm lượng vitamin B1, C và B2 cũng rất khá.
Cà rốt là cây rau ăn củ, năng suất củ giảm mạnh nếu giai đoạn phình củ thiếu nước. Cà rốt là cây chịu lạnh, ưu thích nhiệt độ 16 - 210C, tuy nhiên có thể chịu được nhiệt độ cao 25 - 270C, ở nhiệt độ thích hợp củ phát triển to, n

### Split sentences

In [41]:
import re
from typing import List, Optional

def split_into_sentences(
    text: str,
    min_sentence_length: int = 10,
    max_sentence_length: int = 1000,
    preserve_acronyms: bool = True,
) -> List[str]:
    """
    Split text into sentences, handling edge cases like:
    - Acronyms (U.S.A., Dr., Mr., etc.)
    - Numbers with decimals/periods (3.14, $1.99)
    - Abbreviations (e.g., i.e., vs.)
    - URLs and email addresses
    - Quotation marks and parentheses
    - Bullet points and list items
    """
    if not text or not text.strip():
        return []
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text.strip())
    
    # Protect abbreviations from being split
    abbreviations = [
        r'\b(?:Mr|Mrs|Ms|Miss|Dr|Prof|Sr|Jr|St|Ave|Blvd|Dept|Univ|Corp|Inc|Ltd|Co)\.',
        r'\b(?:etc|vs|approx|dept|est|govt|natl|orig|temp)\.',
        r'\b(?:Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\.',
        r'\b[A-Z]\.(?:\s[A-Z]\.)*',  # U.S.A., F.B.I.
    ]
    
    # Placeholder for protected terms
    protected = {}
    counter = [0]
    
    def protect_pattern(pattern: str) -> str:
        def _protect(match: re.Match) -> str:
            placeholder = f"__PROTECTED_{counter[0]}__"
            protected[placeholder] = match.group()
            counter[0] += 1
            return placeholder
        return _protect
    
    for abbr in abbreviations:
        text = re.sub(abbr, protect_pattern(abbr), text)
    
    # Handle numbers with decimals
    text = re.sub(r'\d+\.\d+', protect_pattern(r'\d+\.\d+'), text)
    
    # Handle URLs and emails
    url_pattern = r'https?://[^\s<>"]+|www\.[^\s<>"]+'
    text = re.sub(url_pattern, protect_pattern(url_pattern), text)
    email_pattern = r'[\w.+-]+@[\w-]+\.[\w.-]+'
    text = re.sub(email_pattern, protect_pattern(email_pattern), text)
    
    # Handle ellipsis
    text = text.replace('...', '__ELLIPSIS__')
    
    # Main sentence splitting pattern
    # Split on: . ! ? followed by whitespace and capital letter or end of string
    sentence_end = r'(?<=[.?!])\s+(?=[A-Z"\'«»„“])'
    sentences = re.split(sentence_end, text)
    
    # Handle cases where sentence end is followed by lowercase (e.g., quotes)
    secondary_split = r'(?<=[.?!])\s+(?=[«»„""''""])'
    processed_sentences = []
    for sent in sentences:
        subs = re.split(secondary_split, sent)
        processed_sentences.extend(subs)
    
    # Handle newlines and list items
    final_sentences = []
    for sent in processed_sentences:
        # Split on newlines if they seem like separate sentences
        parts = re.split(r'\n\s*\n', sent)
        for part in parts:
            part = part.strip()
            if part:
                # Handle bullet points and numbered lists
                list_items = re.split(r'\n(?=\s*[\-\*\•]|\d+\.\s)', part)
                final_sentences.extend([li.strip() for li in list_items if li.strip()])
    
    # Restore protected patterns
    restored_sentences = []
    for sent in final_sentences:
        sent = sent.replace('__ELLIPSIS__', '...')
        for placeholder, original in protected.items():
            sent = sent.replace(placeholder, original)
        restored_sentences.append(sent)
    
    # Post-processing
    cleaned_sentences = []
    for sent in restored_sentences:
        # Remove leading/trailing whitespace and quotes
        sent = sent.strip().strip('"').strip("'").strip('«').strip('»')
        sent = re.sub(r'\s+', ' ', sent)
        
        # Filter based on length
        if len(sent) >= min_sentence_length and len(sent) <= max_sentence_length:
            cleaned_sentences.append(sent)
        elif len(sent) > max_sentence_length:
            # For very long sentences, try to split on commas/semicolons
            sub_sentences = re.split(r'(?<=[,;])\s+', sent)
            cleaned_sentences.extend(sub_sentences)
    
    return cleaned_sentences if cleaned_sentences else [text]


# Alternative: Using spaCy for more accurate sentence splitting
def split_sentences_spacy(text: str, model: str = "en_core_web_sm") -> List[str]:
    """Use spaCy's sentence segmentation (more accurate but heavier)."""
    import spacy
    
    try:
        nlp = spacy.load(model)
    except OSError:
        import subprocess
        subprocess.run(["python", "-m", "spacy", "download", model])
        nlp = spacy.load(model)
    
    # Disable unnecessary pipeline components for speed
    nlp.disable_pipe_names()  # Disable all
    nlp.enable_pipe("senter")  # Only enable sentence segmentation
    
    doc = nlp(text[:1000000])  # Limit to 1M chars for memory
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]


In [46]:
input_df_1.columns

Index(['text', 'id', 'domain'], dtype='str')

In [45]:
sentences = split_into_sentences(input_df_2['text'][1000])
print(f"Found {len(sentences)} sentences:")
for i, s in enumerate(sentences, 1):
    print(f"{i}: {s}")

Found 22 sentences:
1: Bón NPK-S Lâm Thao cho cây cà rốt?
2: Chủ nhật, 18/03/2018 05:04 GMT+7 10/12/2013, 11:40 (GMT+7) Để tạo năng suất như trong SX, cây cà rốt lấy đi từ đất lượng đạm thuộc loại nhiều (chia làm 5 nhóm rau, cây cà rốt ở nhóm đầu), kali và lân thuộc loại trung bình thấp (nhóm thứ 3 của 4 nhóm).
3: Cây cà rốt có tên khoa học là Daucus carota var. sativa thuộc họ Hoa tán Umbelliferae.
4: Cây cà rốt được trồng đầu tiên ở Afghanistan và sau đó được trồng ở 10 - 12 nước khu vực phía Đông vùng Địa Trung Hải, rồi chuyển ra phía Tây khoảng 14 - 15 nước Châu Âu.
5: Cây cà rốt được sử dụng từ lâu, đầu tiên được coi là cây thuốc, mãi cho đến những năm đầu thế kỷ XX mới được sử dụng làm cây thực phẩm.
6: Cà rốt nổi tiếng vì là nguồn dồi dào vitaminA, hàm lượng vitamin B1, C và B2 cũng rất khá.
7: Cà rốt là cây rau ăn củ, năng suất củ giảm mạnh nếu giai đoạn phình củ thiếu nước.
8: Cà rốt là cây chịu lạnh, ưu thích nhiệt độ 16 - 210C, tuy nhiên có thể chịu được nhiệt độ cao 25 - 27

In [ ]:
print(input_df_1['text'][0])

### Sentence EDA - See how many sentences and decide the max sentence limit

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_splitter import split_into_sentences
import warnings
warnings.filterwarnings('ignore')

# Load all parquet files
def load_parquet_dataset(
    folder_path: str, 
    sample: int = None  # Set to 5000 for quick analysis
) -> pd.DataFrame:
    """Load all parquet files from folder."""
    import glob
    import os
    
    parquet_files = glob.glob(os.path.join(folder_path, "*.parquet"))
    print(f"Found {len(parquet_files)} parquet files")
    
    dfs = []
    for f in tqdm(parquet_files, desc="Loading parquet files"):
        df = pd.read_parquet(f)
        dfs.append(df)
    
    df = pd.concat(dfs, ignore_index=True)
    print(f"Total rows: {len(df):,}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"Label distribution:\n{df['domain'].value_counts()}")
    
    if sample:
        df = df.sample(sample, random_state=42)
        print(f"Sampled to {sample} rows")
    
    return df


# Analyze sentence distribution for your specific dataset
def analyze_parquet_dataset(df: pd.DataFrame):
    """Full analysis optimized for 100k parquet dataset."""
    
    texts = df['text'].tolist()
    labels = df['domain'].tolist()
    
    print(f"\n{'='*70}")
    print(f"📊 ANALYZING {len(texts):,} NEWS ARTICLES")
    print(f"{'='*70}")
    
    print(f"\nLabel distribution:")
    label_counts = df['domain'].value_counts()
    for label, count in label_counts.items():
        print(f"  {label}: {count:,} ({count/len(df)*100:.1f}%)")
    
    print(f"\nText length stats (characters):")
    text_lengths = df['text'].str.len()
    print(f"  Mean:    {text_lengths.mean():>10,.0f}")
    print(f"  Median:  {text_lengths.median():>10,.0f}")
    print(f"  Min:     {text_lengths.min():>10,}")
    print(f"  Max:     {text_lengths.max():>10,}")
    print(f"  P90:     {text_lengths.quantile(0.9):>10,.0f}")
    print(f"  P95:     {text_lengths.quantile(0.95):>10,.0f}")
    print(f"  P99:     {text_lengths.quantile(0.99):>10,.0f}")
    
    # Sentence splitting (might take a few minutes for 100k)
    print(f"\n🔄 Splitting {len(texts):,} texts into sentences (this may take a few minutes)...")
    
    from concurrent.futures import ProcessPoolExecutor, as_completed
    import multiprocessing
    
    num_workers = multiprocessing.cpu_count()
    print(f"Using {num_workers} CPU cores")
    
    # Process in parallel
    def split_text(text):
        return split_into_sentences(text)
    
    all_sentences = []
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(split_text, text): i for i, text in enumerate(texts)}
        
        for future in tqdm(as_completed(futures), total=len(texts), desc="Splitting sentences"):
            all_sentences.append(future.result())
    
    # # Alternative: Serial processing (slower but simpler)
    # all_sentences = []
    # for text in tqdm(texts, desc="Splitting sentences"):
    #     all_sentences.append(split_into_sentences(text))
    
    # Statistics
    sentence_counts = [len(s) for s in all_sentences]
    word_counts = []
    for sentences in all_sentences:
        for sent in sentences:
            word_counts.append(len(sent.split()))
    
    sentence_counts = np.array(sentence_counts)
    word_counts = np.array(word_counts)
    
    print(f"\n{'='*70}")
    print(f"📈 SENTENCE DISTRIBUTION RESULTS")
    print(f"{'='*70}")
    
    print(f"\n📄 Sentences per Document:")
    print(f"  Mean:    {sentence_counts.mean():>8.1f}")
    print(f"  Median:  {np.median(sentence_counts):>8.0f}")
    print(f"  Std:     {sentence_counts.std():>8.1f}")
    print(f"  Min:     {sentence_counts.min():>8}")
    print(f"  Max:     {sentence_counts.max():>8}")
    print(f"  P25:     {np.percentile(sentence_counts, 25):>8.0f}")
    print(f"  P50:     {np.percentile(sentence_counts, 50):>8.0f}")
    print(f"  P75:     {np.percentile(sentence_counts, 75):>8.0f}")
    print(f"  P90:     {np.percentile(sentence_counts, 90):>8.0f}")
    print(f"  P95:     {np.percentile(sentence_counts, 95):>8.0f}")
    print(f"  P98:     {np.percentile(sentence_counts, 98):>8.0f}")
    print(f"  P99:     {np.percentile(sentence_counts, 99):>8.0f}")
    
    print(f"\n📝 Words per Sentence:")
    print(f"  Mean:    {word_counts.mean():>8.1f}")
    print(f"  Median:  {np.median(word_counts):>8.0f}")
    print(f"  Min:     {word_counts.min():>8}")
    print(f"  Max:     {word_counts.max():>8}")
    print(f"  P90:     {np.percentile(word_counts, 90):>8.0f}")
    print(f"  P95:     {np.percentile(word_counts, 95):>8.0f}")
    print(f"  P99:     {np.percentile(word_counts, 99):>8.0f}")
    
    # Distribution breakdown
    total = len(sentence_counts)
    print(f"\n📊 Distribution of Documents by Sentence Count:")
    print(f"  0 sentences:     {np.sum(sentence_counts == 0):>6} ({np.sum(sentence_counts == 0)/total*100:.1f}%)")
    print(f"  1-5 sentences:   {np.sum((sentence_counts >= 1) & (sentence_counts <= 5)):>6} ({np.sum((sentence_counts >= 1) & (sentence_counts <= 5))/total*100:.1f}%)")
    print(f"  6-10 sentences:  {np.sum((sentence_counts >= 6) & (sentence_counts <= 10)):>6} ({np.sum((sentence_counts >= 6) & (sentence_counts <= 10))/total*100:.1f}%)")
    print(f"  11-20 sentences: {np.sum((sentence_counts >= 11) & (sentence_counts <= 20)):>6} ({np.sum((sentence_counts >= 11) & (sentence_counts <= 20))/total*100:.1f}%)")
    print(f"  21-40 sentences: {np.sum((sentence_counts >= 21) & (sentence_counts <= 40)):>6} ({np.sum((sentence_counts >= 21) & (sentence_counts <= 40))/total*100:.1f}%)")
    print(f"  41-60 sentences: {np.sum((sentence_counts >= 41) & (sentence_counts <= 60)):>6} ({np.sum((sentence_counts >= 41) & (sentence_counts <= 60))/total*100:.1f}%)")
    print(f"  61-100 sentences:{np.sum((sentence_counts >= 61) & (sentence_counts <= 100)):>6} ({np.sum((sentence_counts >= 61) & (sentence_counts <= 100))/total*100:.1f}%)")
    print(f"  >100 sentences:  {np.sum(sentence_counts > 100):>6} ({np.sum(sentence_counts > 100)/total*100:.1f}%)")
    
    # Recommendations
    print(f"\n{'='*70}")
    print(f"💡 RECOMMENDED HAN PARAMETERS")
    print(f"{'='*70}")
    
    p95_sentences = int(np.percentile(sentence_counts, 95))
    p90_sentences = int(np.percentile(sentence_counts, 90))
    p95_words = int(np.percentile(word_counts, 95))
    
    if p95_sentences <= 40:
        print(f"\n✅ max_sentences = 40")
        print(f"   Covers {p95_sentences} sentences (95th percentile)")
    elif p90_sentences <= 40:
        print(f"\n⚖️  Option 1: max_sentences = 40 (covers {np.sum(sentence_counts <= 40)/total*100:.0f}% of data)")
        print(f"   Option 2: max_sentences = {p95_sentences} (covers 95% of data)")
    else:
        print(f"\n⚠️  Long documents detected! Options:")
        print(f"   Option 1: max_sentences = 40 (covers {np.sum(sentence_counts <= 40)/total*100:.0f}% of data)")
        print(f"   Option 2: max_sentences = {p90_sentences} (covers 90% of data)")
        print(f"   Option 3: Sliding window (chunk >{p90_sentences} sentence docs)")
        print(f"   Option 4: Truncate to first N sentences + summarize tail")
    
    # For words, use smaller of P95 or 100
    recommended_words = min(p95_words, 100)
    print(f"\n✅ max_words_per_sentence = {recommended_words}")
    
    return all_sentences, sentence_counts, word_counts

